In [ ]:
# from google.colab import files
# import os

# # Create the local directory in Colab
# os.makedirs('/content/15_Min', exist_ok=True)

# print("Please upload your .xlsx files from your local '15_Min' folder:")
# uploaded = files.upload()

# # Move uploaded files to the directory
# for filename in uploaded.keys():
#     os.rename(filename, os.path.join('/content/15_Min', filename))

# print("Upload complete.")

In [ ]:
%%capture
# Install Unsloth and dependencies
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.29" "trl<0.13.0" peft accelerate bitsandbytes

In [ ]:
from unsloth import FastLanguageModel
import torch
import gc

# Force clear all memory
if "model" in locals(): del model
if "tokenizer" in locals(): del tokenizer
if "trainer" in locals(): del trainer
gc.collect()
torch.cuda.empty_cache()

max_seq_length = 256 # Further reduced for T4 stability
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/DeepSeek-R1-Distill-Llama-8B",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 8, # Reduced from 16 to save VRAM
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

print("Model reloaded with peak memory optimizations.")

In [ ]:
!pip install -q transformers accelerate bitsandbytes

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

model_id = "unsloth/DeepSeek-R1-Distill-Llama-8B"

# Configure 4-bit quantization for T4 compatibility
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16
)

print(f"Model {model_id} loaded successfully using Transformers on T4.")

KeyboardInterrupt: 

In [ ]:
!pip install --force-reinstall "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.29" "trl<0.13.0" peft accelerate bitsandbytes
!pip install pyarrow==14.0.1

In [ ]:
import os
import glob
import pandas as pd
import torch
from unsloth import FastLanguageModel

# 1. Load the model for Finetuning
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/DeepSeek-R1-Distill-Llama-8B",
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

### Data Preparation
The following cell will look into your `15_Min` folder. To proceed, I have set up a template that mimics a 'Rule-Based' extraction.

**Note:** Since I cannot read your analyzer's specific code, you should ensure the `get_signal` function below matches the logic in your `Analyser(1).ipynb`.

In [ ]:
def prepare_training_data(folder_path):
    import glob
    import os
    all_files = glob.glob(os.path.join(folder_path, "*.xlsx"))
    dataset = []

    for file in all_files:
        try:
            df = pd.read_excel(file)
            # Take the last 5 rows to provide context for the model
            context_data = df.tail(5).to_string()
            prompt = f"Analyze the following stock data and provide a signal based on the fixed parameters: {context_data}"

            # This is where your actual Analyser logic should label the data
            # For now, we use a placeholder
            label = "BUY"

            dataset.append({"instruction": "Predict Buy/Sell signal", "input": prompt, "output": label})
        except Exception as e:
            print(f"Skipping {file} due to error: {e}")

    return pd.DataFrame(dataset)

training_df = prepare_training_data('/content/15_Min')
display(training_df.head())

### Step 1: Define Rule-Based Logic
Paste the logic from your `Analyser(1).ipynb` here so the model can learn to replicate it exactly.

In [ ]:
from datasets import Dataset

def get_rule_based_label(df):
    # TODO: Replace with your actual logic from Analyser(1).ipynb
    # Example:
    # if df['Close'].iloc[-1] > df['Open'].iloc[-1]: return 'BUY'
    return 'BUY'

data_list = []
for file in glob.glob('/content/15_Min/*.xlsx'):
    try:
        df = pd.read_excel(file)
        label = get_rule_based_label(df)
        # We format the training example: Data context -> Signal
        prompt = f"Instruction: Predict the trade signal based on the fixed parameters.\nInput: {df.tail(10).to_string()}\nOutput: "
        data_list.append({"text": f"{prompt}{label}"})
    except Exception as e:
        print(f"Error processing {file}: {e}")

train_dataset = Dataset.from_pandas(pd.DataFrame(data_list))

### Step 2: Execute Fine-Tuning
This uses the Unsloth SFTTrainer to update the model weights.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    dataset_text_field = "text",
    max_seq_length = 2048,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()

### Define Your Fine-Tuning Logic
In this step, we prepare the dataset for the model. We use your rule-based parameters to create labels for the stock data in the `15_Min` folder.

In [ ]:
from datasets import Dataset

def get_rule_based_label(df):
    # This should be replaced with the exact logic from your Analyser(1).ipynb
    # For example, if it checks for a specific RSI or Moving Average crossover:
    # if df['Close'].iloc[-1] > df['MA'].iloc[-1]: return 'BUY'
    return 'BUY' # Default placeholder

data_list = []
for file in glob.glob('/content/15_Min/*.xlsx'):
    df = pd.read_excel(file)
    label = get_rule_based_label(df)
    # Formatting the prompt for the model
    prompt = f"Instruction: Predict the trade signal based on the fixed parameters.\nInput: {df.tail(5).to_string()}\nOutput: "
    data_list.append({"text": f"{prompt}{label}"})

dataset = Dataset.from_pandas(pd.DataFrame(data_list))

### Run Fine-Tuning
The following code starts the training process using Unsloth's optimized trainer.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 2048,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()

### Define Your Rules
Paste your logic from `Analyser(1).ipynb` into the `get_rule_based_signal` function below. This ensures the model learns your exact fixed parameters.

In [ ]:
import glob
import os
import pandas as pd
import numpy as np
from datasets import Dataset

def get_rule_based_signal(df):
    try:
        # Ensure columns are handled regardless of case
        df.columns = [c.capitalize() for c in df.columns]
        if 'Close' not in df.columns: return 'HOLD'

        period = 50
        multiplier = 3.0

        df['EMA'] = df['Close'].ewm(span=period, adjust=False).mean()
        df['AbsDiff'] = df['Close'].diff().abs()
        df['AvgRange'] = df['AbsDiff'].ewm(span=period, adjust=False).mean()
        df['SmoothRange'] = df['AvgRange'].ewm(span=(period * 2 - 1), adjust=False).mean() * multiplier

        df['Filter'] = df['Close'].copy()
        for i in range(1, len(df)):
            prev_f = df.at[df.index[i-1], 'Filter']
            sr = df.at[df.index[i], 'SmoothRange']
            curr_c = df.at[df.index[i], 'Close']
            if curr_c > prev_f:
                df.at[df.index[i], 'Filter'] = max(prev_f, curr_c - sr)
            else:
                df.at[df.index[i], 'Filter'] = min(prev_f, curr_c + sr)

        last_idx = df.index[-1]
        prev_idx = df.index[-2]
        if df.at[last_idx, 'Filter'] > df.at[prev_idx, 'Filter']:
            return 'BUY'
        elif df.at[last_idx, 'Filter'] < df.at[prev_idx, 'Filter']:
            return 'SELL'
        return 'HOLD'
    except:
        return 'HOLD'

def prepare_final_dataset(folder_path, tokenizer):
    all_files = glob.glob(os.path.join(folder_path, '*.xlsx'))
    data_list = []
    for file in all_files:
        try:
            df = pd.read_excel(file)
            df.columns = [c.capitalize() for c in df.columns]
            if len(df) < 55 or 'Close' not in df.columns: continue

            label = get_rule_based_signal(df.copy())
            context = df[['Close']].tail(10).to_string()
            text = f'### Instruction:\nPredict the trend signal (BUY/SELL/HOLD) based on Smooth Range Filter logic.\n\n### Input:\n{context}\n\n### Response:\n{label}'
            data_list.append({'text': text})
        except Exception as e:
            pass

    if not data_list:
        raise ValueError("No valid data found. Please check if Excel files contain a 'Close' column.")

    dataset = Dataset.from_pandas(pd.DataFrame(data_list))
    return dataset.map(lambda x: tokenizer(x['text'], truncation=True, padding='max_length', max_length=256), batched=True, remove_columns=['text'])

train_dataset = prepare_final_dataset('/content/15_Min', tokenizer)

In [ ]:
import json

def extract_logic_from_notebook(nb_path):
    with open(nb_path, 'r', encoding='utf-8') as f:
        nb = json.load(f)

    print('--- Content of Analyser(1).ipynb ---')
    for cell in nb['cells']:
        if cell['cell_type'] == 'code':
            content = ''.join(cell['source'])
            if 'RSI' in content or 'signal' in content or 'close' in content:
                print(f'Possible Signal Logic Found:\n{content}\n---')

extract_logic_from_notebook('/content/Analyser(1).ipynb')

### Start Fine-tuning
This cell initializes the SFTTrainer to train the model on your signals.

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling
import torch
import gc

# Clearing memory before the final run
gc.collect()
torch.cuda.empty_cache()

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model = model,
    train_dataset = train_dataset,
    data_collator = data_collator,
    args = TrainingArguments(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, # Increased steps for better learning of the new logic
        learning_rate = 2e-4,
        fp16 = True,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
        gradient_checkpointing = True,
        remove_unused_columns = False,
        max_grad_norm = 0.3
    ),
)

trainer_stats = trainer.train()
print('Fine-tuning complete with Analyser(1) logic.')

In [ ]:
model.save_pretrained("fine_tuned_stock_model")
tokenizer.save_pretrained("fine_tuned_stock_model")
print("Model saved to 'fine_tuned_stock_model' folder.")

In [ ]:
import torch

# Test on a single file
test_file = '/content/15_Min/EICHERMOT_15min.xlsx'
df_test = pd.read_excel(test_file)

# Prepare the prompt exactly like training
prompt = f"### Instruction:\nPredict the trade signal based on the fixed parameters.\n\n### Input:\n{df_test.tail(5).to_string()}\n\n### Response:\n"

inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=10)
prediction = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("--- Model Prediction ---")
print(prediction.split("### Response:")[-1].strip())

### Step 1: Install Hugging Face Hub

In [ ]:
!pip install --upgrade huggingface_hub

### Step 2: Login to Hugging Face
When you run the cell below, a login widget will appear. Paste your **Write Access Token** from Hugging Face.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
from huggingface_hub import notebook_login, logout, whoami

# Force logout to clear any cached READ tokens
try:
    logout()
    print('Logged out of previous session.')
except:
    pass

print('Now, please login using a token with "WRITE" permissions:')
notebook_login()

# Verification step
try:
    user = whoami()
    if 'accessToken' in user and user['auth']['type'] == 'access_token':
        print(f'\nSuccessfully logged in as: {user["name"]}')
        print('Token verified. Please try running the upload cell again.')
    else:
        print('\nWarning: Your login might not have full write permissions.')
except Exception as e:
    print(f'Verification failed: {e}')

Now, please login using a token with "WRITE" permissions:



In [ ]:
import getpass
from huggingface_hub import HfApi, login

print("Please paste your Hugging Face WRITE token below:")
hf_token = getpass.getpass()

try:
    login(token=hf_token, add_to_git_credential=True)
    api = HfApi()
    user = api.whoami()
    print(f"\nSuccessfully authenticated as: {user['name']}")
    print("You should now be able to run the model upload cell (1aea8bb9) successfully.")
except Exception as e:
    print(f"\nAuthentication failed: {e}")

Please paste your Hugging Face WRITE token below:


KeyboardInterrupt: Interrupted by user

### 🛑 How to get your Hugging Face WRITE Token

1. Log in to [Hugging Face](https://huggingface.co/).
2. Go to **Settings** -> **[Access Tokens](https://huggingface.co/settings/tokens)**.
3. Click **+ Create new token**.
4. **CRITICAL:** Set the **Token type** to **Write** (not Read).
5. Give it a name (e.g., `ColabUpload`) and click **Create token**.
6. Copy that string and paste it into the `getpass` box in the cell above.

### Step 3: Push Model to Hub
Replace `'your-username/your-model-name'` with your desired repository name (e.g., `'john_doe/deepseek-stock-v1'`).

In [ ]:
from huggingface_hub import whoami

try:
    # 1. Fetch current logged-in user information
    user_info = whoami()
    username = user_info["name"]
    hf_repo_id = f"{username}/DeepSeek-R1-Distill-Llama-Finance-8B"
    print(f"Target Repository: {hf_repo_id}")

    # 2. Push model and tokenizer
    # This requires a 'WRITE' token
    model.push_to_hub(hf_repo_id)
    tokenizer.push_to_hub(hf_repo_id)

    print(f"Model successfully pushed to: https://huggingface.co/{hf_repo_id}")
except Exception as e:
    print(f"Error: {e}")
    print("\nTroubleshooting steps:")
    print("1. Ensure you used a 'WRITE' token in notebook_login()")
    print("2. Check if you are actually logged in by running: !huggingface-cli whoami")

Target Repository: Adit251/DeepSeek-R1-Distill-Llama-Finance-8B
Error: (Request ID: Root=1-69e0f358-10a5e9320ea36d5341880030;3673ffa4-cd4a-489c-ae37-d262f1f7c706)

403 Forbidden: You don't have the rights to create a model under the namespace "Adit251".
Cannot access content at: https://huggingface.co/api/repos/create.
Make sure your token has the correct permissions.

Troubleshooting steps:
1. Ensure you used a 'WRITE' token in notebook_login()
2. Check if you are actually logged in by running: !huggingface-cli whoami


In [ ]:
from huggingface_hub import HfApi, create_repo, whoami, repo_exists

try:
    api = HfApi()
    user_info = whoami()
    username = user_info['name']
    repo_id = f'{username}/DeepSeek-R1-Distill-Llama-Finance-8B'

    print(f'Target Repository: {repo_id}')

    # Check if repo exists manually since exists_ok failed
    if not repo_exists(repo_id=repo_id, repo_type='model'):
        print(f'Creating repository: {repo_id}')
        create_repo(repo_id=repo_id, repo_type='model')
    else:
        print(f'Repository {repo_id} already exists.')

    print('Uploading model and tokenizer...')
    model.push_to_hub(repo_id)
    tokenizer.push_to_hub(repo_id)

    print(f'Done! View your model at: https://huggingface.co/{repo_id}')
except Exception as e:
    print(f'Error: {e}')
    print('\nVerification Checklist:')
    print('1. Go to https://huggingface.co/settings/tokens')
    print('2. Ensure your token type is "WRITE".')
    print('3. Re-run the notebook_login() cell with that specific token.')

Target Repository: Adit251/DeepSeek-R1-Distill-Llama-Finance-8B
Creating repository: Adit251/DeepSeek-R1-Distill-Llama-Finance-8B
Error: (Request ID: Root=1-69e0f60d-794eb9d872bace7c5332c684;b85794cd-c61f-4127-9e80-4c348ef41533)

403 Forbidden: You don't have the rights to create a model under the namespace "Adit251".
Cannot access content at: https://huggingface.co/api/repos/create.
Make sure your token has the correct permissions.

Verification Checklist:
1. Go to https://huggingface.co/settings/tokens
2. Ensure your token type is "WRITE".
3. Re-run the notebook_login() cell with that specific token.


In [ ]:
import getpass
from huggingface_hub import HfApi, login, create_repo, whoami, repo_exists

# 1. Authenticate again to ensure 'WRITE' access is active
print("Please paste your Hugging Face WRITE token:")
hf_token = getpass.getpass()

try:
    login(token=hf_token, add_to_git_credential=True)
    user_info = whoami()
    username = user_info['name']
    repo_id = f'{username}/DeepSeek-R1-Distill-Llama-Finance-8B'

    print(f'\nAuthenticated as: {username}')
    print(f'Target Repository: {repo_id}')

    # 2. Ensure repository exists
    if not repo_exists(repo_id=repo_id, repo_type='model'):
        print(f'Creating new repository: {repo_id}')
        create_repo(repo_id=repo_id, repo_type='model')
    else:
        print(f'Repository already exists.')

    # 3. Push the local model and tokenizer
    print('Uploading model weights and tokenizer... This may take a few minutes.')
    model.push_to_hub(repo_id)
    tokenizer.push_to_hub(repo_id)

    print(f'\nSuccess! Your model is live at: https://huggingface.co/{repo_id}')

except Exception as e:
    print(f'\nAn error occurred: {e}')
    print('Check if your token type is definitely set to "WRITE" in Hugging Face settings.')

Please paste your Hugging Face WRITE token:


KeyboardInterrupt: Interrupted by user

In [ ]:
import os

def get_dir_size(path='fine_tuned_stock_model'):
    total = 0
    with os.scandir(path) as it:
        for entry in it:
            if entry.is_file():
                total += entry.stat().st_size
            elif entry.is_dir():
                total += get_dir_size(entry.path)
    return total

size_mb = get_dir_size() / (1024 * 1024)
print(f"Total model size to upload: {size_mb:.2f} MB")
print("If this is over 100MB, it is normal for the upload to take a few minutes.")

Total model size to upload: 96.48 MB
If this is over 100MB, it is normal for the upload to take a few minutes.


In [ ]:
try:
    # Using commit_message to force a fresh transaction
    print(f"Starting optimized upload to {repo_id}...")
    model.push_to_hub(
        repo_id,
        commit_message="Upload fine-tuned stock model",
        private=False
    )
    tokenizer.push_to_hub(repo_id)
    print(f"\nSuccess! Check progress at: https://huggingface.co/{repo_id}/tree/main")
except Exception as e:
    print(f"Upload interrupted or failed: {e}")

Starting optimized upload to Adit251/DeepSeek-R1-Distill-Llama-Finance-8B...


README.md:   0%|          | 0.00/595 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   6%|5         | 5.02MB / 83.9MB            

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpiomi8trx/tokenizer.json: 100%|##########| 17.2MB / 17.2MB            

  ...mpiomi8trx/tokenizer.json:  87%|########6 | 14.9MB / 17.2MB            


Success! Check progress at: https://huggingface.co/Adit251/DeepSeek-R1-Distill-Llama-Finance-8B/tree/main
